In [0]:
if spark.catalog.tableExists("oc_projeto.`oc-tabelas`.gold_af_arr"):
    max_ts = spark.sql("""
        SELECT max(data_carregamento) as ts FROM oc_projeto.`oc-tabelas`.gold_af_arr
    """).collect()[0]["ts"]
else:
    max_ts = None



In [0]:
if max_ts is None:
    df_silver_af_arr = spark.table("oc_projeto.`oc-tabelas`.silver_voos_arr")
else:
    df_silver_af_arr = spark.table("oc_projeto.`oc-tabelas`.silver_voos_arr").filter(f"data_carregamento > '{max_ts}'")



In [0]:
df_silver_af_arr.createOrReplaceTempView("temp_af_arr")

df_gold_af_arr = spark.sql("""
    SELECT
        id_voos as id_af_arr,
        data,
        arr_voo,
        arr_iata,
        dep_iata_dep,
        destino_af,
        turno,
        
        coalesce(arr_qtd, 0) AS qtd_af,
        data_carregamento
    FROM temp_af_arr             
""")

In [0]:
from pyspark.sql.functions import current_timestamp

df_gold_af_arr = df_gold_af_arr.withColumn("data_process_gold", current_timestamp())

In [0]:
if not spark.catalog.tableExists("oc_projeto.`oc-tabelas`.gold_af_arr"):
    df_gold_af_arr.write.format("delta").saveAsTable("oc_projeto.`oc-tabelas`.gold_af_arr")
else:
    df_gold_af_arr.createOrReplaceTempView("temp_gold_af_arr")

    spark.sql("""
          MERGE INTO oc_projeto.`oc-tabelas`.gold_af_arr AS t
          USING temp_gold_af_arr AS s
          ON t.id_af_arr = s.id_af_arr
          WHEN NOT MATCHED THEN INSERT *
          """)
print("Dados carregados com sucesso na gold_af_arr com MERGE")